# UPI Shield AI — Notebook 4: Model Training

## Objective

The objective of this notebook is to train machine learning models for the UPI Shield AI transaction risk scoring system.

In Notebook 3, we created a RAM-safe processed dataset and saved it in Parquet format.

In this notebook, we will:

1. Load the processed dataset.
2. Separate input features and target column.
3. Split the dataset into training and testing sets.
4. Apply SMOTE only on the training data to handle class imbalance.
5. Train multiple machine learning models.
6. Compare model performance using fraud detection metrics.
7. Save the trained models and best model.

## Important Note on SMOTE

SMOTE should be applied only after train-test split and only on training data.

If SMOTE is applied before train-test split, synthetic fraud samples may leak into the test set and produce unrealistic model performance.

In [1]:
# ---------------------------------------------------------
# Step 1: Import Required Libraries
# ---------------------------------------------------------
# We import libraries required for:
# 1. Data loading
# 2. Model training
# 3. SMOTE imbalance handling
# 4. Model evaluation
# 5. Saving trained models

import os
import json
import warnings
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ---------------------------------------------------------
# Step 2: Mount Google Drive
# ---------------------------------------------------------
# We mount Google Drive to access processed data and save models.

from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [3]:
# ---------------------------------------------------------
# Step 3: Define Project Paths
# ---------------------------------------------------------
# We define all important project paths.

BASE_DIR = "/content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring"

PROCESSED_DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
MODELS_DIR = os.path.join(BASE_DIR, "models")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")

PROCESSED_DATA_PATH = os.path.join(PROCESSED_DATA_DIR, "processed_fraud_data.parquet")
FEATURE_LIST_PATH = os.path.join(PROCESSED_DATA_DIR, "feature_list.json")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("Processed Data Path:", PROCESSED_DATA_PATH)
print("Feature Metadata Path:", FEATURE_LIST_PATH)
print("Models Directory:", MODELS_DIR)
print("Reports Directory:", REPORTS_DIR)

Processed Data Path: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/data/processed/processed_fraud_data.parquet
Feature Metadata Path: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/data/processed/feature_list.json
Models Directory: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/models
Reports Directory: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/reports


In [4]:
# ---------------------------------------------------------
# Step 4: Check Required Files
# ---------------------------------------------------------
# Before loading, we check whether required files exist.

if os.path.exists(PROCESSED_DATA_PATH):
    print("Processed Parquet dataset found.")
else:
    print("Processed dataset not found. Please check Notebook 3 output.")

if os.path.exists(FEATURE_LIST_PATH):
    print("Feature metadata file found.")
else:
    print("Feature metadata file not found. Please check Notebook 3 output.")

Processed Parquet dataset found.
Feature metadata file found.


In [5]:
# ---------------------------------------------------------
# Step 5: Load Processed Dataset
# ---------------------------------------------------------
# We load the processed dataset saved in Parquet format.
# Parquet is faster and more RAM-efficient than CSV.

df = pd.read_parquet(PROCESSED_DATA_PATH)

print("Processed dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Processed dataset loaded successfully.
Shape: (208213, 25)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balanceDiffOrig,balanceDiffDest,errorBalanceOrig,errorBalanceDest,isZeroBalanceAfterTransaction,isHighAmount,hourOfDay,isHighRiskType,logAmount,logBalanceDiffOrig,logBalanceDiffDest,logErrorBalanceOrig,logErrorBalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,349,65487.9883,141478.0000,75990.0078,0.0000,0.0000,0,65487.9922,0.0000,0.0078,65487.9883,0,0,13,0,11.0896,11.0896,0.0000,0.0078,11.0896,0,0,0,1,0
1,257,299940.0312,0.0000,0.0000,502460.1250,802400.1250,0,0.0000,299940.0000,-299940.0312,0.0000,1,0,17,1,12.6113,0.0000,12.6113,12.6113,0.0000,0,1,0,0,0
2,281,16878.8594,17579.6504,700.7900,0.0000,0.0000,0,16878.8613,0.0000,0.0010,16878.8594,0,0,17,0,9.7339,9.7339,0.0000,0.0010,9.7339,0,0,0,1,0
3,42,132800.6250,0.0000,0.0000,638717.3125,771517.9375,0,0.0000,132800.6250,-132800.6250,0.0000,1,0,18,1,11.7966,0.0000,11.7966,11.7966,0.0000,0,1,0,0,0
4,15,238788.1562,0.0000,0.0000,672467.0625,911255.2500,0,0.0000,238788.1875,-238788.1562,0.0000,1,0,15,1,12.3833,0.0000,12.3833,12.3833,0.0000,0,1,0,0,0


In [6]:
# ---------------------------------------------------------
# Step 6: Load Feature Metadata
# ---------------------------------------------------------
# Feature metadata contains:
# 1. Final feature names
# 2. Target column name
# 3. High amount threshold
# 4. Processed file information

with open(FEATURE_LIST_PATH, "r") as file:
    feature_metadata = json.load(file)

features = feature_metadata["features"]
target = feature_metadata["target"]

print("Feature metadata loaded successfully.")
print("Target column:", target)
print("Total features:", len(features))

print("\nFirst 10 features:")
features[:10]

Feature metadata loaded successfully.
Target column: isFraud
Total features: 24

First 10 features:


['step',
 'amount',
 'oldbalanceOrg',
 'newbalanceOrig',
 'oldbalanceDest',
 'newbalanceDest',
 'balanceDiffOrig',
 'balanceDiffDest',
 'errorBalanceOrig',
 'errorBalanceDest']

In [7]:
# ---------------------------------------------------------
# Step 7: Basic Dataset Check
# ---------------------------------------------------------
# We check target distribution before training.

print("Dataset shape:", df.shape)

print("\nTarget distribution:")
print(df[target].value_counts())

print("\nTarget percentage:")
print(df[target].value_counts(normalize=True) * 100)

print("\nMissing values:", df.isnull().sum().sum())

Dataset shape: (208213, 25)

Target distribution:
isFraud
0    200000
1      8213
Name: count, dtype: int64

Target percentage:
isFraud
0   96.0555
1    3.9445
Name: proportion, dtype: float64

Missing values: 0


In [8]:
# ---------------------------------------------------------
# Step 8: Separate Features and Target
# ---------------------------------------------------------
# X contains input features.
# y contains output target.
#
# Target:
# 0 = Normal transaction
# 1 = Fraud transaction

X = df[features]
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (208213, 24)
y shape: (208213,)

Target distribution:
isFraud
0    200000
1      8213
Name: count, dtype: int64


In [9]:
# ---------------------------------------------------------
# Step 9: Free Extra Memory
# ---------------------------------------------------------
# After creating X and y, we delete the original dataframe
# to reduce RAM usage.

del df

import gc
gc.collect()

print("Original dataframe deleted from memory.")

Original dataframe deleted from memory.


In [10]:
# ---------------------------------------------------------
# Step 10: Train-Test Split
# ---------------------------------------------------------
# We split the data into training and testing sets.
#
# stratify=y ensures that fraud/normal ratio is preserved
# in both train and test sets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train-test split completed.")

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Train-test split completed.

X_train shape: (166570, 24)
X_test shape: (41643, 24)
y_train shape: (166570,)
y_test shape: (41643,)

Training target distribution:
isFraud
0    160000
1      6570
Name: count, dtype: int64

Testing target distribution:
isFraud
0    40000
1     1643
Name: count, dtype: int64


## Why SMOTE Is Applied After Train-Test Split

Fraud detection datasets are highly imbalanced. To handle this, we use SMOTE, which creates synthetic minority class samples.

However, SMOTE must be applied only on the training data.

If SMOTE is applied before train-test split, synthetic fraud examples can leak into the test set. This causes data leakage and gives unrealistic model performance.

Correct approach:

`Train-test split → Apply SMOTE on training data only → Train model → Evaluate on original test data`

In [11]:
# ---------------------------------------------------------
# Step 11: Apply SMOTE on Training Data Only
# ---------------------------------------------------------
# SMOTE creates synthetic fraud samples to reduce class imbalance.
#
# We apply SMOTE only on X_train and y_train.
# We do NOT apply SMOTE on test data.
#
# sampling_strategy=0.5 means:
# minority class will become 50% of majority class.
#
# This is safer than making both classes fully equal because
# full balancing can sometimes create too many synthetic samples.

smote = SMOTE(
    sampling_strategy=0.5,
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("SMOTE applied successfully on training data only.")

print("\nBefore SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

print("\nX_train_smote shape:", X_train_smote.shape)
print("y_train_smote shape:", y_train_smote.shape)

SMOTE applied successfully on training data only.

Before SMOTE:
isFraud
0    160000
1      6570
Name: count, dtype: int64

After SMOTE:
isFraud
0    160000
1     80000
Name: count, dtype: int64

X_train_smote shape: (240000, 24)
y_train_smote shape: (240000,)


In [12]:
# ---------------------------------------------------------
# Step 12: Define Model Evaluation Function
# ---------------------------------------------------------
# This function calculates important classification metrics.
#
# For fraud detection, recall and F1-score are very important
# because fraud cases are rare and missing fraud is costly.

def evaluate_model(model_name, model, X_test_data, y_test_data):
    """
    Evaluates a trained classification model.

    Parameters:
    model_name: Name of the model
    model: Trained model object
    X_test_data: Test features
    y_test_data: True test labels

    Returns:
    Dictionary containing evaluation metrics
    """

    y_pred = model.predict(X_test_data)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test_data)[:, 1]
        roc_auc = roc_auc_score(y_test_data, y_proba)
    else:
        y_proba = None
        roc_auc = np.nan

    cm = confusion_matrix(y_test_data, y_pred)

    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test_data, y_pred),
        "Precision": precision_score(y_test_data, y_pred, zero_division=0),
        "Recall": recall_score(y_test_data, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test_data, y_pred, zero_division=0),
        "ROC AUC": roc_auc,
        "True Negatives": tn,
        "False Positives": fp,
        "False Negatives": fn,
        "True Positives": tp
    }

    return metrics

In [13]:
# ---------------------------------------------------------
# Step 13: Create Storage for Models and Results
# ---------------------------------------------------------
# We will store trained models, scalers, and evaluation results.

trained_models = {}
model_results = []

print("Model storage created.")

Model storage created.


In [14]:
# ---------------------------------------------------------
# Step 14: Train Logistic Regression
# ---------------------------------------------------------
# Logistic Regression is our baseline model.
#
# Since Logistic Regression is sensitive to feature scale,
# we apply StandardScaler.
#
# Scaling is fitted only on training data and then applied
# to test data.

logistic_scaler = StandardScaler()

X_train_smote_scaled = logistic_scaler.fit_transform(X_train_smote)
X_test_scaled = logistic_scaler.transform(X_test)

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

logistic_model.fit(X_train_smote_scaled, y_train_smote)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [15]:
# ---------------------------------------------------------
# Step 15: Evaluate Logistic Regression
# ---------------------------------------------------------
# We evaluate Logistic Regression on original test data.

logistic_metrics = evaluate_model(
    "Logistic Regression",
    logistic_model,
    X_test_scaled,
    y_test
)

model_results.append(logistic_metrics)

trained_models["Logistic Regression"] = {
    "model": logistic_model,
    "scaler": logistic_scaler,
    "requires_scaling": True
}

logistic_metrics

{'Model': 'Logistic Regression',
 'Accuracy': 0.9995197272050524,
 'Precision': 0.992116434202547,
 'Recall': 0.995739500912964,
 'F1 Score': 0.9939246658566221,
 'ROC AUC': np.float64(0.9995447808886183),
 'True Negatives': np.int64(39987),
 'False Positives': np.int64(13),
 'False Negatives': np.int64(7),
 'True Positives': np.int64(1636)}

In [16]:
# ---------------------------------------------------------
# Step 16: Train Decision Tree
# ---------------------------------------------------------
# Decision Tree learns rule-based patterns.
# It is easy to explain but can overfit if not controlled.

decision_tree_model = DecisionTreeClassifier(
    max_depth=12,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    class_weight="balanced"
)

decision_tree_model.fit(X_train_smote, y_train_smote)

print("Decision Tree trained successfully.")

Decision Tree trained successfully.


In [17]:
# ---------------------------------------------------------
# Step 17: Evaluate Decision Tree
# ---------------------------------------------------------
# We evaluate Decision Tree on original test data.

decision_tree_metrics = evaluate_model(
    "Decision Tree",
    decision_tree_model,
    X_test,
    y_test
)

model_results.append(decision_tree_metrics)

trained_models["Decision Tree"] = {
    "model": decision_tree_model,
    "scaler": None,
    "requires_scaling": False
}

decision_tree_metrics

{'Model': 'Decision Tree',
 'Accuracy': 0.9993756453665682,
 'Precision': 0.9867549668874173,
 'Recall': 0.9975654290931223,
 'F1 Score': 0.9921307506053268,
 'ROC AUC': np.float64(0.99890905356056),
 'True Negatives': np.int64(39978),
 'False Positives': np.int64(22),
 'False Negatives': np.int64(4),
 'True Positives': np.int64(1639)}

In [18]:
# ---------------------------------------------------------
# Step 18: Train Random Forest
# ---------------------------------------------------------
# Random Forest combines multiple decision trees.
# It usually performs well on tabular datasets.

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

random_forest_model.fit(X_train_smote, y_train_smote)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [19]:
# ---------------------------------------------------------
# Step 19: Evaluate Random Forest
# ---------------------------------------------------------
# We evaluate Random Forest on original test data.

random_forest_metrics = evaluate_model(
    "Random Forest",
    random_forest_model,
    X_test,
    y_test
)

model_results.append(random_forest_metrics)

trained_models["Random Forest"] = {
    "model": random_forest_model,
    "scaler": None,
    "requires_scaling": False
}

random_forest_metrics

{'Model': 'Random Forest',
 'Accuracy': 0.9998319045217684,
 'Precision': 0.998780487804878,
 'Recall': 0.996956786366403,
 'F1 Score': 0.997867803837953,
 'ROC AUC': np.float64(0.9998127054169201),
 'True Negatives': np.int64(39998),
 'False Positives': np.int64(2),
 'False Negatives': np.int64(5),
 'True Positives': np.int64(1638)}

In [20]:
# ---------------------------------------------------------
# Step 20: Train XGBoost
# ---------------------------------------------------------
# XGBoost is a strong boosting algorithm and works very well
# on structured/tabular data.
#
# We use moderate parameters to avoid Colab RAM issues.

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_model.fit(X_train_smote, y_train_smote)

print("XGBoost trained successfully.")

XGBoost trained successfully.


In [21]:
# ---------------------------------------------------------
# Step 21: Evaluate XGBoost
# ---------------------------------------------------------
# We evaluate XGBoost on original test data.

xgb_metrics = evaluate_model(
    "XGBoost",
    xgb_model,
    X_test,
    y_test
)

model_results.append(xgb_metrics)

trained_models["XGBoost"] = {
    "model": xgb_model,
    "scaler": None,
    "requires_scaling": False
}

xgb_metrics

{'Model': 'XGBoost',
 'Accuracy': 0.9997598636025262,
 'Precision': 0.9975624619134674,
 'Recall': 0.9963481436396835,
 'F1 Score': 0.9969549330085262,
 'ROC AUC': np.float64(0.9996224893487523),
 'True Negatives': np.int64(39996),
 'False Positives': np.int64(4),
 'False Negatives': np.int64(6),
 'True Positives': np.int64(1637)}

In [22]:
# ---------------------------------------------------------
# Step 22: Create Model Comparison Table
# ---------------------------------------------------------
# We convert model results into a clean comparison table.

results_df = pd.DataFrame(model_results)

results_df = results_df.sort_values(
    by=["Recall", "F1 Score", "ROC AUC"],
    ascending=False
).reset_index(drop=True)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,True Negatives,False Positives,False Negatives,True Positives
0,Decision Tree,0.9994,0.9868,0.9976,0.9921,0.9989,39978,22,4,1639
1,Random Forest,0.9998,0.9988,0.9970,0.9979,0.9998,39998,2,5,1638
2,XGBoost,0.9998,0.9976,0.9963,0.9970,0.9996,39996,4,6,1637
3,Logistic Regression,0.9995,0.9921,0.9957,0.9939,0.9995,39987,13,7,1636


## Model Selection Priority

In fraud detection, accuracy is not the most important metric.

A model can get high accuracy by predicting most transactions as normal, but this is dangerous because it may miss fraud cases.

For this project, model selection priority is:

1. Recall
2. F1-score
3. ROC-AUC
4. Precision
5. Accuracy

Recall is important because it tells how many actual fraud transactions were correctly detected.

In [23]:
# ---------------------------------------------------------
# Step 23: Select Best Model
# ---------------------------------------------------------
# We select the best model based on Recall, F1 Score, and ROC AUC.

best_model_name = results_df.iloc[0]["Model"]

best_model_info = trained_models[best_model_name]

print("Best Model Selected:", best_model_name)

print("\nBest Model Metrics:")
results_df.iloc[0]

Best Model Selected: Decision Tree

Best Model Metrics:


,0
Model,Decision Tree
Accuracy,0.9994
Precision,0.9868
Recall,0.9976
F1 Score,0.9921
ROC AUC,0.9989
True Negatives,39978
False Positives,22
False Negatives,4
True Positives,1639


In [24]:
# ---------------------------------------------------------
# Step 24: Generate Classification Reports
# ---------------------------------------------------------
# We print detailed classification reports for all models.

for model_name, model_info in trained_models.items():
    model = model_info["model"]
    requires_scaling = model_info["requires_scaling"]
    scaler = model_info["scaler"]

    if requires_scaling:
        X_test_for_model = scaler.transform(X_test)
    else:
        X_test_for_model = X_test

    y_pred = model.predict(X_test_for_model)

    print("=" * 70)
    print("Classification Report:", model_name)
    print("=" * 70)
    print(classification_report(y_test, y_pred, target_names=["Normal", "Fraud"]))
    print("\n")

Classification Report: Logistic Regression
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     40000
       Fraud       0.99      1.00      0.99      1643

    accuracy                           1.00     41643
   macro avg       1.00      1.00      1.00     41643
weighted avg       1.00      1.00      1.00     41643



Classification Report: Decision Tree
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     40000
       Fraud       0.99      1.00      0.99      1643

    accuracy                           1.00     41643
   macro avg       0.99      1.00      1.00     41643
weighted avg       1.00      1.00      1.00     41643



Classification Report: Random Forest
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     40000
       Fraud       1.00      1.00      1.00      1643

    accuracy                           1.00     41643
   macr

In [25]:
# ---------------------------------------------------------
# Step 25: Save Model Comparison Report
# ---------------------------------------------------------
# We save the model comparison table as CSV inside reports folder.

MODEL_COMPARISON_PATH = os.path.join(REPORTS_DIR, "04_model_training_comparison.csv")

results_df.to_csv(MODEL_COMPARISON_PATH, index=False)

print("Model comparison report saved successfully.")
print("Saved path:", MODEL_COMPARISON_PATH)

Model comparison report saved successfully.
Saved path: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/reports/04_model_training_comparison.csv


In [26]:
# ---------------------------------------------------------
# Step 26: Save All Trained Models
# ---------------------------------------------------------
# We save every trained model as a joblib file.
# Each saved object contains:
# 1. Model
# 2. Scaler if required
# 3. Whether scaling is required
# 4. Feature names

for model_name, model_info in trained_models.items():
    safe_model_name = model_name.lower().replace(" ", "_")

    model_bundle = {
        "model_name": model_name,
        "model": model_info["model"],
        "scaler": model_info["scaler"],
        "requires_scaling": model_info["requires_scaling"],
        "features": features,
        "target": target,
        "feature_metadata": feature_metadata
    }

    model_path = os.path.join(MODELS_DIR, f"{safe_model_name}_model.pkl")

    joblib.dump(model_bundle, model_path)

    print(f"{model_name} saved successfully at:")
    print(model_path)

Logistic Regression saved successfully at:
/content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/models/logistic_regression_model.pkl
Decision Tree saved successfully at:
/content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/models/decision_tree_model.pkl
Random Forest saved successfully at:
/content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/models/random_forest_model.pkl
XGBoost saved successfully at:
/content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/models/xgboost_model.pkl


In [27]:
# ---------------------------------------------------------
# Step 27: Save Best Model Separately
# ---------------------------------------------------------
# We save the best model separately so the Streamlit app can
# load it directly without checking all models.

best_model_bundle = {
    "model_name": best_model_name,
    "model": best_model_info["model"],
    "scaler": best_model_info["scaler"],
    "requires_scaling": best_model_info["requires_scaling"],
    "features": features,
    "target": target,
    "feature_metadata": feature_metadata,
    "model_selection_metrics": results_df.iloc[0].to_dict()
}

BEST_MODEL_PATH = os.path.join(MODELS_DIR, "best_model.pkl")

joblib.dump(best_model_bundle, BEST_MODEL_PATH)

print("Best model saved successfully.")
print("Best model:", best_model_name)
print("Saved path:", BEST_MODEL_PATH)

Best model saved successfully.
Best model: Decision Tree
Saved path: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/models/best_model.pkl


In [28]:
# ---------------------------------------------------------
# Step 28: Verify Best Model Loading
# ---------------------------------------------------------
# We reload the best model to confirm that it was saved properly.

loaded_best_model = joblib.load(BEST_MODEL_PATH)

print("Best model loaded successfully.")
print("Model name:", loaded_best_model["model_name"])
print("Requires scaling:", loaded_best_model["requires_scaling"])
print("Number of features:", len(loaded_best_model["features"]))

Best model loaded successfully.
Model name: Decision Tree
Requires scaling: False
Number of features: 24


In [29]:
# ---------------------------------------------------------
# Step 29: Test Prediction on Few Samples
# ---------------------------------------------------------
# We test the saved best model on a few records from test data.

sample_X = X_test.head(10)
sample_y = y_test.head(10)

model = loaded_best_model["model"]
scaler = loaded_best_model["scaler"]
requires_scaling = loaded_best_model["requires_scaling"]

if requires_scaling:
    sample_X_model = scaler.transform(sample_X)
else:
    sample_X_model = sample_X

sample_predictions = model.predict(sample_X_model)

if hasattr(model, "predict_proba"):
    sample_probabilities = model.predict_proba(sample_X_model)[:, 1]
else:
    sample_probabilities = [None] * len(sample_predictions)

sample_results = pd.DataFrame({
    "Actual": sample_y.values,
    "Predicted": sample_predictions,
    "Fraud Probability": sample_probabilities
})

sample_results["Actual Label"] = sample_results["Actual"].map({
    0: "Normal",
    1: "Fraud"
})

sample_results["Predicted Label"] = sample_results["Predicted"].map({
    0: "Normal",
    1: "Fraud"
})

sample_results

,Actual,Predicted,Fraud Probability,Actual Label,Predicted Label
0,0,0,0.0000,Normal,Normal
1,0,0,0.0000,Normal,Normal
2,0,0,0.0000,Normal,Normal
3,0,0,0.0000,Normal,Normal
4,0,0,0.0000,Normal,Normal
5,0,0,0.0000,Normal,Normal
6,0,0,0.0000,Normal,Normal
7,0,0,0.0000,Normal,Normal
8,0,0,0.0007,Normal,Normal
9,0,0,0.0000,Normal,Normal


In [30]:
# ---------------------------------------------------------
# Step 30: Save Training Summary Report
# ---------------------------------------------------------
# We save a text report containing training details,
# SMOTE strategy, model comparison, and best model selection.

training_report_path = os.path.join(REPORTS_DIR, "04_model_training_report.txt")

with open(training_report_path, "w") as file:
    file.write("UPI Shield AI — Model Training Report\n")
    file.write("=" * 55 + "\n\n")

    file.write("Dataset Used:\n")
    file.write(f"{PROCESSED_DATA_PATH}\n\n")

    file.write("Feature Count:\n")
    file.write(f"{len(features)}\n\n")

    file.write("Train-Test Split:\n")
    file.write("Test Size: 20%\n")
    file.write("Random State: 42\n")
    file.write("Stratified Split: Yes\n\n")

    file.write("Training Target Distribution Before SMOTE:\n")
    file.write(str(y_train.value_counts()))
    file.write("\n\n")

    file.write("Training Target Distribution After SMOTE:\n")
    file.write(str(y_train_smote.value_counts()))
    file.write("\n\n")

    file.write("SMOTE Strategy:\n")
    file.write("Applied only on training data\n")
    file.write("sampling_strategy = 0.5\n\n")

    file.write("Models Trained:\n")
    for model_name in trained_models.keys():
        file.write(f"- {model_name}\n")

    file.write("\nModel Comparison:\n")
    file.write(str(results_df))
    file.write("\n\n")

    file.write("Best Model Selected:\n")
    file.write(best_model_name)
    file.write("\n\n")

    file.write("Best Model Metrics:\n")
    file.write(str(results_df.iloc[0]))
    file.write("\n\n")

    file.write("Saved Files:\n")
    file.write("- logistic_regression_model.pkl\n")
    file.write("- decision_tree_model.pkl\n")
    file.write("- random_forest_model.pkl\n")
    file.write("- xgboost_model.pkl\n")
    file.write("- best_model.pkl\n")
    file.write("- 04_model_training_comparison.csv\n")

print("Model training report saved successfully.")
print("Report path:", training_report_path)

Model training report saved successfully.
Report path: /content/drive/MyDrive/UPI-Shield-AI-Transaction-Risk-Scoring/reports/04_model_training_report.txt


In [31]:
# ---------------------------------------------------------
# Step 31: Final Notebook Summary
# ---------------------------------------------------------
# This confirms that Notebook 4 has been completed.

print("Notebook 4: Model Training completed successfully.")

print("\nKey Outputs:")
print("- Processed Parquet dataset loaded")
print("- Feature metadata loaded")
print("- Train-test split completed")
print("- SMOTE applied only on training data")
print("- Logistic Regression trained")
print("- Decision Tree trained")
print("- Random Forest trained")
print("- XGBoost trained")
print("- Model comparison report saved")
print("- Best model selected and saved")
print("- Training summary report saved")

print("\nBest Model:", best_model_name)

print("\nNext Notebook:")
print("05_model_evaluation.ipynb")

Notebook 4: Model Training completed successfully.

Key Outputs:
- Processed Parquet dataset loaded
- Feature metadata loaded
- Train-test split completed
- SMOTE applied only on training data
- Logistic Regression trained
- Decision Tree trained
- Random Forest trained
- XGBoost trained
- Model comparison report saved
- Best model selected and saved
- Training summary report saved

Best Model: Decision Tree

Next Notebook:
05_model_evaluation.ipynb


# Notebook 4 Summary

In this notebook, we trained multiple machine learning models for UPI Shield AI.

## Models Trained

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

## Imbalance Handling

SMOTE was applied only on the training data after train-test split.

This prevents data leakage and ensures that the test data remains original and unseen.

## Evaluation Metrics Used

1. Accuracy
2. Precision
3. Recall
4. F1-score
5. ROC-AUC
6. Confusion Matrix
7. False Negatives

## Best Model

The best model was selected based mainly on:

1. Recall
2. F1-score
3. ROC-AUC

## Output Files Created

1. Trained model files in `models/`
2. `best_model.pkl`
3. `04_model_training_comparison.csv`
4. `04_model_training_report.txt`

## Next Step

The next notebook will be:

`05_model_evaluation.ipynb`

In Notebook 5, we will deeply evaluate the best model, create confusion matrix plots, analyze fraud probability, test custom transactions, and prepare the model for the Streamlit app.